# Predicting Student Test Scores 
## Score: 8.56431

In [5]:
import os
import hashlib
import numpy as np
import pandas as pd
import xgboost as xgb
import warnings

from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
from sklearn.preprocessing import TargetEncoder

warnings.filterwarnings('ignore')
np.random.seed(42)

In [6]:
TRAIN_PATH = 'playground-series-s6e1/train.csv'
TEST_PATH = 'playground-series-s6e1/test.csv'
ORIGINAL_PATH = 'Exam_Score_Prediction.csv'

TARGET = 'exam_score'
ID_COL = 'id'

N_FOLDS = 10
RANDOM_STATE = 80085

if not os.path.exists(ORIGINAL_PATH):
    raise FileNotFoundError(f'Original dataset not found at {ORIGINAL_PATH}')

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
original_df = pd.read_csv(ORIGINAL_PATH)

print(f'Train:    {train_df.shape}')
print(f'Test:     {test_df.shape}')
print(f'Original: {original_df.shape}')

base_features = [c for c in train_df.columns if c not in [TARGET, ID_COL]]
cat_features = train_df.select_dtypes('object').columns.tolist()

print(f'\nBase features: {len(base_features)}')
print(f'Categorical:   {cat_features}')

Train:    (630000, 13)
Test:     (270000, 12)
Original: (20000, 13)

Base features: 11
Categorical:   ['gender', 'course', 'internet_access', 'sleep_quality', 'study_method', 'facility_rating', 'exam_difficulty']


In [7]:
def engineer_features(df, base_cols):
    out = df.copy()
    eps = 1e-5
    
    study = out['study_hours'].clip(lower=0)
    attend = out['class_attendance'].clip(lower=0)
    sleep = out['sleep_hours'].clip(lower=0)
    
    out['study_hours_squared'] = out['study_hours'] ** 2
    out['class_attendance_squared'] = out['class_attendance'] ** 2
    out['sleep_hours_squared'] = out['sleep_hours'] ** 2
    out['age_squared'] = out['age'] ** 2
    
    out['log_study_hours'] = np.log1p(study)
    out['log_class_attendance'] = np.log1p(attend)
    out['log_sleep_hours'] = np.log1p(sleep)
    
    out['sqrt_study_hours'] = np.sqrt(study)
    out['sqrt_class_attendance'] = np.sqrt(attend)
    
    out['study_hours_times_attendance'] = out['study_hours'] * out['class_attendance']
    out['study_hours_times_sleep'] = out['study_hours'] * out['sleep_hours']
    out['attendance_times_sleep'] = out['class_attendance'] * out['sleep_hours']
    out['age_times_study_hours'] = out['age'] * out['study_hours']
    
    out['study_hours_over_sleep'] = out['study_hours'] / (out['sleep_hours'] + eps)
    out['attendance_over_sleep'] = out['class_attendance'] / (out['sleep_hours'] + eps)
    out['attendance_over_study'] = out['class_attendance'] / (out['study_hours'] + eps)
    
    ordinal_maps = {
        'sleep_quality': {'poor': 0, 'average': 1, 'good': 2},
        'facility_rating': {'low': 0, 'medium': 1, 'high': 2},
        'exam_difficulty': {'easy': 0, 'moderate': 1, 'hard': 2}
    }
    for col, mapping in ordinal_maps.items():
        out[f'{col}_numeric'] = out[col].map(mapping).fillna(1).astype(int)
    
    out['study_hours_times_sleep_quality'] = out['study_hours'] * out['sleep_quality_numeric']
    out['attendance_times_facility'] = out['class_attendance'] * out['facility_rating_numeric']
    out['sleep_hours_times_difficulty'] = out['sleep_hours'] * out['exam_difficulty_numeric']
    out['facility_x_sleepq'] = out['facility_rating_numeric'] * out['sleep_quality_numeric']
    out['difficulty_x_facility'] = out['exam_difficulty_numeric'] * out['facility_rating_numeric']
    
    out['high_att_high_study'] = ((out['class_attendance'] >= 90) & (out['study_hours'] >= 6)).astype(int)
    out['ideal_sleep_flag'] = ((out['sleep_hours'] >= 7) & (out['sleep_hours'] <= 9)).astype(int)
    out['high_study_flag'] = (out['study_hours'] >= 7).astype(int)
    
    out['efficiency'] = (out['study_hours'] * out['class_attendance']) / (out['sleep_hours'] + 1)
    
    out['sleep_gap_8'] = (out['sleep_hours'] - 8.0).abs()
    out['attendance_gap_100'] = (out['class_attendance'] - 100.0).abs()
    
    out['study_bin_num'] = pd.cut(out['study_hours'], bins=5, labels=False).astype(int)
    out['attendance_bin_num'] = pd.cut(out['class_attendance'], bins=5, labels=False).astype(int)
    out['sleep_bin_num'] = pd.cut(out['sleep_hours'], bins=5, labels=False).astype(int)
    out['age_bin_num'] = pd.cut(out['age'], bins=5, labels=False).astype(int)
    
    engineered_cols = [
        'study_hours_squared', 'class_attendance_squared', 'sleep_hours_squared', 'age_squared',
        'log_study_hours', 'log_class_attendance', 'log_sleep_hours',
        'sqrt_study_hours', 'sqrt_class_attendance',
        'study_hours_times_attendance', 'study_hours_times_sleep', 'attendance_times_sleep',
        'age_times_study_hours',
        'study_hours_over_sleep', 'attendance_over_sleep', 'attendance_over_study',
        'sleep_quality_numeric', 'facility_rating_numeric', 'exam_difficulty_numeric',
        'study_hours_times_sleep_quality', 'attendance_times_facility', 'sleep_hours_times_difficulty',
        'facility_x_sleepq', 'difficulty_x_facility',
        'high_att_high_study', 'ideal_sleep_flag', 'high_study_flag',
        'efficiency',
        'sleep_gap_8', 'attendance_gap_100',
        'study_bin_num', 'attendance_bin_num', 'sleep_bin_num', 'age_bin_num'
    ]
    
    return out[base_cols + engineered_cols], engineered_cols

X_train, engineered_cols = engineer_features(train_df, base_features)
X_test, _ = engineer_features(test_df, base_features)
X_orig, _ = engineer_features(original_df, base_features)

y_train = train_df[TARGET].reset_index(drop=True)
y_orig = original_df[TARGET].reset_index(drop=True)

full_data = pd.concat([X_train, X_test, X_orig], axis=0, ignore_index=True)
for col in engineered_cols:
    full_data[col] = full_data[col].astype(float)

n_train, n_test = len(train_df), len(test_df)
X = full_data.iloc[:n_train].copy()
X_test = full_data.iloc[n_train:n_train + n_test].copy()
X_original = full_data.iloc[n_train + n_test:].copy()

print(f'Engineered features: {len(engineered_cols)}')
print(f'Total features:      {X.shape[1]} (11 base + {len(engineered_cols)} engineered)')

Engineered features: 34
Total features:      45 (11 base + 34 engineered)


In [8]:
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

oof_ridge = np.zeros(len(X))
test_preds_ridge = np.zeros((len(X_test), N_FOLDS))
orig_preds_ridge = np.zeros(len(X_original))

ridge_alphas = np.logspace(-3, 3, 20)

print('Training Ridge Regression')
print('-' * 40)

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y_train), 1):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    X_tr_aug = pd.concat([X_tr, X_original], axis=0)
    y_tr_aug = pd.concat([y_tr, y_orig], axis=0)
    
    encoder = TargetEncoder(smooth='auto', target_type='continuous')
    X_tr_enc = X_tr_aug.copy()
    X_val_enc = X_val.copy()
    X_test_enc = X_test.copy()
    
    X_tr_enc[cat_features] = encoder.fit_transform(X_tr_aug[cat_features], y_tr_aug)
    X_val_enc[cat_features] = encoder.transform(X_val[cat_features])
    X_test_enc[cat_features] = encoder.transform(X_test[cat_features])
    
    ridge = RidgeCV(alphas=ridge_alphas, cv=5, scoring='neg_root_mean_squared_error')
    ridge.fit(X_tr_enc, y_tr_aug.values.ravel())
    
    oof_ridge[val_idx] = np.clip(ridge.predict(X_val_enc), 0, 100)
    test_preds_ridge[:, fold - 1] = np.clip(ridge.predict(X_test_enc), 0, 100)
    orig_preds_ridge += np.clip(ridge.predict(X_tr_enc.iloc[-len(X_original):]), 0, 100) / N_FOLDS
    
    rmse = np.sqrt(mean_squared_error(y_val, oof_ridge[val_idx]))
    print(f'Fold {fold:2d} | RMSE: {rmse:.6f}')

ridge_oof_rmse = np.sqrt(mean_squared_error(y_train, oof_ridge))
print(f'\nRidge OOF RMSE: {ridge_oof_rmse:.6f}')

Training Ridge Regression
----------------------------------------


Fold  1 | RMSE: 8.869829
Fold  2 | RMSE: 8.886225
Fold  3 | RMSE: 8.850582
Fold  4 | RMSE: 8.929161
Fold  5 | RMSE: 8.907212
Fold  6 | RMSE: 8.883147
Fold  7 | RMSE: 8.914281
Fold  8 | RMSE: 8.900490
Fold  9 | RMSE: 8.895808
Fold 10 | RMSE: 8.895259

Ridge OOF RMSE: 8.893225


In [9]:
for col in base_features:
    full_data[col] = full_data[col].astype(str).astype('category')
for col in engineered_cols:
    full_data[col] = full_data[col].astype(float)

X_xgb = full_data.iloc[:n_train].copy()
X_test_xgb = full_data.iloc[n_train:n_train + n_test].copy()
X_orig_xgb = full_data.iloc[n_train + n_test:].copy()

X_xgb['ridge_pred'] = oof_ridge
X_test_xgb['ridge_pred'] = test_preds_ridge.mean(axis=1)
X_orig_xgb['ridge_pred'] = orig_preds_ridge

print(f'Final feature count: {X_xgb.shape[1]} (including Ridge meta-feature)')

Final feature count: 46 (including Ridge meta-feature)


In [10]:
xgb_params = {
    'n_estimators': 20000,
    'learning_rate': 0.004,
    'max_depth': 9,
    'subsample': 0.78,
    'colsample_bytree': 0.55,
    'colsample_bynode': 0.65,
    'reg_lambda': 6,
    'reg_alpha': 0.15,
    'min_child_weight': 6,
    'tree_method': 'hist',
    'enable_categorical': True,
    'eval_metric': 'rmse',
    'early_stopping_rounds': 100,
    'random_state': 42
}

test_preds_xgb = []
oof_xgb = np.zeros(len(X_xgb))

print('Training XGBoost')
print('-' * 40)

for fold, (train_idx, val_idx) in enumerate(kf.split(X_xgb, y_train), 1):
    print(f'\nFold {fold}/{N_FOLDS}')
    
    X_tr, X_val = X_xgb.iloc[train_idx], X_xgb.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    X_tr_aug = pd.concat([X_tr, X_orig_xgb], axis=0)
    y_tr_aug = pd.concat([y_tr, y_orig], axis=0)
    
    model = xgb.XGBRegressor(**xgb_params)
    model.fit(
        X_tr_aug, y_tr_aug,
        eval_set=[(X_val, y_val)],
        verbose=1000
    )
    
    oof_xgb[val_idx] = model.predict(X_val)
    test_preds_xgb.append(model.predict(X_test_xgb))
    
    rmse = np.sqrt(mean_squared_error(y_val, oof_xgb[val_idx]))
    print(f'Validation RMSE: {rmse:.5f}')

xgb_oof_rmse = np.sqrt(mean_squared_error(y_train, oof_xgb))
print(f'\nXGBoost OOF RMSE: {xgb_oof_rmse:.5f}')

Training XGBoost
----------------------------------------

Fold 1/10
[0]	validation_0-rmse:18.94164
[1000]	validation_0-rmse:8.60492
[2000]	validation_0-rmse:8.57721
[2176]	validation_0-rmse:8.57725
Validation RMSE: 8.57704

Fold 2/10
[0]	validation_0-rmse:18.78986
[1000]	validation_0-rmse:8.61581
[2000]	validation_0-rmse:8.58972
[2461]	validation_0-rmse:8.58897
Validation RMSE: 8.58882

Fold 3/10
[0]	validation_0-rmse:18.84631
[1000]	validation_0-rmse:8.59561
[2000]	validation_0-rmse:8.56685
[2363]	validation_0-rmse:8.56634
Validation RMSE: 8.56616

Fold 4/10
[0]	validation_0-rmse:18.85472
[1000]	validation_0-rmse:8.67148
[2000]	validation_0-rmse:8.64832
[2306]	validation_0-rmse:8.64801
Validation RMSE: 8.64784

Fold 5/10
[0]	validation_0-rmse:18.86911
[1000]	validation_0-rmse:8.65621
[2000]	validation_0-rmse:8.63401
[2216]	validation_0-rmse:8.63402
Validation RMSE: 8.63390

Fold 6/10
[0]	validation_0-rmse:18.86549
[1000]	validation_0-rmse:8.62706
[1944]	validation_0-rmse:8.60634
Vali

In [11]:
print('Model Performance')
print('-' * 40)
print(f'Ridge OOF RMSE:   {ridge_oof_rmse:.6f}')
print(f'XGBoost OOF RMSE: {xgb_oof_rmse:.5f}')

print(f'\nFeature Summary')
print('-' * 40)
print(f'Base features:       {len(base_features)}')
print(f'Engineered features: {len(engineered_cols)}')
print(f'Meta-feature:        1')
print(f'Total:               {X_xgb.shape[1]}')

Model Performance
----------------------------------------
Ridge OOF RMSE:   8.893225
XGBoost OOF RMSE: 8.60836

Feature Summary
----------------------------------------
Base features:       11
Engineered features: 34
Meta-feature:        1
Total:               46


In [12]:
test_ids = test_df[ID_COL].values
final_preds = np.clip(np.mean(test_preds_xgb, axis=0), 0, 100)

submission = pd.DataFrame({ID_COL: test_ids, TARGET: final_preds})
submission.to_csv('submission.csv', index=False)

with open('submission.csv', 'rb') as f:
    md5 = hashlib.md5(f.read()).hexdigest()

print('submission.csv')
print('md5', md5)
print(f'pred_mean {final_preds.mean():.4f}')
print(f'pred_std {final_preds.std():.4f}')
print(f'pred_min {final_preds.min():.4f}')
print(f'pred_max {final_preds.max():.4f}')

submission.csv
md5 d13663fb861cabd2fac6e07541c5215e
pred_mean 62.5471
pred_std 16.7853
pred_min 16.7384
pred_max 100.0000
